# 02. Features por Fuente

## Goal
Create features from leads and project data before building the final clustering matrix.


## Inputs
- `outputs/base_empresas_analitica.parquet`
- Raw fields from leads and project files linked through the final entity key.

## Outputs
- `outputs/features_empresas.parquet`
- `outputs/features_empresas.csv`


## Suggested Feature Families
- Leads coverage and completeness.
- Company duplication intensity in leads.
- Project or horas intensity by company.
- Variety of records, categories, or time windows.
- Missingness flags that can help clustering.
- Source consistency flags between leads and horas.


In [ ]:

# ── Helpers y rutas ──────────────────────────────────────────────────────────
from pathlib import Path
import re, unicodedata, pickle
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    def display(x): print(x)

def find_project_root(start=None):
    start = (start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "01_data_ingestion_enrichment").is_dir() and (p / "02_data_cleaning").is_dir():
            return p
    raise FileNotFoundError("No se pudo localizar la raíz.")

def ensure_dir(d): Path(d).mkdir(parents=True, exist_ok=True); return Path(d)

def save_df_csv(df, path, *, index=False, encoding="utf-8-sig"):
    path = Path(path); ensure_dir(path.parent)
    df.to_csv(path, index=index, encoding=encoding)
    print(f"[OK] Guardado: {path.resolve()}  shape: {df.shape}"); return path

ROOT       = find_project_root()
INGESTION  = ROOT / "01_data_ingestion_enrichment"
OUTPUT_DIR = ROOT / "03_feature_engineering" / "outputs"
ensure_dir(OUTPUT_DIR)

print(f"[CONFIG] ROOT      : {ROOT}")
print(f"[CONFIG] OUTPUT_DIR: {OUTPUT_DIR}")

# ── Cargar base analítica ─────────────────────────────────────────────────────
_base_pkl = OUTPUT_DIR / "base_empresas_analitica.pkl"
_base_csv = OUTPUT_DIR / "base_empresas_analitica.csv"
if _base_pkl.exists():
    with open(_base_pkl, "rb") as f: base = pickle.load(f)
    print(f"[BASE] cargado desde pkl  {base.shape}")
elif _base_csv.exists():
    base = pd.read_csv(_base_csv)
    print(f"[BASE] cargado desde csv  {base.shape}")
else:
    raise FileNotFoundError(f"No existe la base analítica en {OUTPUT_DIR}")

# ── Cargar fuentes raw ────────────────────────────────────────────────────────
def _strip_accents(s): return "".join(ch for ch in unicodedata.normalize("NFKD",str(s)) if not unicodedata.combining(ch))
def normalize_company_name(value):
    if value is None or (isinstance(value,float) and np.isnan(value)): return ""
    s = _strip_accents(str(value).strip()).upper()
    s = re.sub(r"[^A-Z0-9 ]+"," ",s)
    s = re.sub(r"\b(SA|S\s*A|S\.A\.?|S\.A\.S\.?|SAS|LTDA|CIA|C\.?IA\.?|COMPANIA|COMPAÑIA|CORP|INC|LLC|C\.L\.?|C\.?LTDA\.?|\&|Y)\b"," ",s)
    s = re.sub(r"\b(DE|DEL|LA|EL|LOS|LAS)\b"," ",s)
    return re.sub(r"\s+", " ", s).strip()

_leads_path = next((p for p in [INGESTION/"leads.xlsx", ROOT/"leads.xlsx"] if p.exists()), None)
_horas_path = next((p for p in [INGESTION/"proyectos_empresa.xlsx", ROOT/"proyectos_empresa.xlsx"] if p.exists()), None)

if _leads_path is None: raise FileNotFoundError("No encontré leads.xlsx")
if _horas_path is None: raise FileNotFoundError("No encontré proyectos_empresa.xlsx")

df_leads = pd.read_excel(_leads_path)
df_horas  = pd.read_excel(_horas_path)
df_leads["_name_norm"] = df_leads["Company"].map(normalize_company_name)
df_horas["_name_norm"]  = df_horas["EMPRESA"].map(normalize_company_name)
print(f"[LEADS] {df_leads.shape}  [HORAS] {df_horas.shape}")

# ── Bloque 1: features_leads ──────────────────────────────────────────────────
_total_leads_cols = df_leads.shape[1]
features_leads = (
    df_leads.groupby("_name_norm", as_index=False)
    .agg(fl_n_rows=("_name_norm", "size"))
    .rename(columns={"_name_norm": "name_norm"})
)
_completeness = (
    df_leads.assign(_completeness=df_leads.notna().sum(axis=1) / _total_leads_cols)
    .groupby("_name_norm", as_index=False)["_completeness"]
    .mean()
    .rename(columns={"_name_norm":"name_norm","_completeness":"fl_avg_completeness"})
)
features_leads = features_leads.merge(_completeness, on="name_norm", how="left")
features_leads["fl_duplication_ratio"] = features_leads["fl_n_rows"] / max(1, len(features_leads))

# ── Bloque 2: features_horas ──────────────────────────────────────────────────
fh_agg = {"fh_n_rows": ("_name_norm", "size")}
num_cols_horas = df_horas.select_dtypes(include="number").columns.tolist()
for c in num_cols_horas[:5]:
    fh_agg[f"fh_sum_{c}"]  = (c, "sum")
    fh_agg[f"fh_mean_{c}"] = (c, "mean")

features_horas = (
    df_horas.groupby("_name_norm", as_index=False)
    .agg(**fh_agg)
    .rename(columns={"_name_norm": "name_norm"})
)

# ── Bloque 3: quality_flags ───────────────────────────────────────────────────
quality_flags = base[["name_norm","RUC","source_in_leads","source_in_horas","verdict"]].copy()
quality_flags["qf_has_ruc"]       = quality_flags["RUC"].notna() & (quality_flags["RUC"].astype(str).str.len() == 13)
quality_flags["qf_in_both"]       = quality_flags["source_in_leads"] & quality_flags["source_in_horas"]
quality_flags["qf_match_correct"] = quality_flags["verdict"].isin(["correct","SCVS_EXACT"])
quality_flags = quality_flags.drop(columns=["RUC","source_in_leads","source_in_horas","verdict"])

# ── Merge final ───────────────────────────────────────────────────────────────
features_empresas = (
    base[["entity_id","name_norm","canonical_name"]]
    .merge(features_leads, on="name_norm", how="left")
    .merge(features_horas,  on="name_norm", how="left")
    .merge(quality_flags,   on="name_norm", how="left")
)
num_feat_cols = features_empresas.select_dtypes(include="number").columns
features_empresas[num_feat_cols] = features_empresas[num_feat_cols].fillna(0)

print(f"\n[FEATURES] shape: {features_empresas.shape}")
print(f"  Columnas: {features_empresas.columns.tolist()}")

_ = save_df_csv(features_empresas, OUTPUT_DIR / "features_empresas.csv")
with open(OUTPUT_DIR / "features_empresas.pkl", "wb") as f: pickle.dump(features_empresas, f)
print(f"[OK] features_empresas.pkl guardado")

# ── Verificación ──────────────────────────────────────────────────────────────
for p in [OUTPUT_DIR/"features_empresas.csv", OUTPUT_DIR/"features_empresas.pkl"]:
    if not p.exists(): raise FileNotFoundError(f"Output faltante: {p}")
    print(f"[OK] {p.name}")
print("\n✓ Notebook 02_features_fuentes completado correctamente.")
display(features_empresas.head(10))
